In [6]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# organizei frases diferentes em grupos semanticos
# grupo A: frases sobre cachorro (mesmo significado palavras diferentes)
# grupo B: frases sobre tecnologia
# grupo C: frases sem contexto

# criando uma lista
frases = [
    #grupo A
    'O cachorro correu pelo parque',
    'O cão foi embora correndo pelo jardim',
    'Um cão brincou no quintal',

    # grupo B
    'Inteligência artificial está tranformando o mundo',
    'Machine learning é uma área da IA',
    'Redes neurais aprendem com dados',

    # grupo C
    'A receita de bolo leva farinhas e ovos'
]

labels = ["cachorro-0", "cachorro-1", "cachorro-2", "tech-3", "tech-4", "tech-5", "comida-6"]

print("Frases carregadas:", len(frases))

Frases carregadas: 7


In [4]:
# BoW aqui cada frase vira um vetor onde cada posição é uma palavra do vocabulario total,
# e o valor é a contagem daquela palavra - Porem BoW inogra ordem, contexto e sinonimos

bow_vectorizer = CountVectorizer() # cria o objeto "transformador de texto em contagem de palavras". Nao sabe nada ainda sobre a frase
x_bow = bow_vectorizer.fit_transform(frases) # fit lê todas as frases e monta o vocabulário e transform converte cada frase num vetor

# vamos ver o vocbulario que foi aprendido
vocabulario = bow_vectorizer.get_feature_names_out() # Retorna uma lista com todas as palavras únicas encontradas em ordem alfabética.
print(f"Tamanho do vacbulario Bow: {len(vocabulario)} palavras únicas") # Exibe quantas palavras únicas existem no conjunto total de frases
print(f"Shape da matriz Bow: {x_bow.shape} ({len(frases)} frases x {len(vocabulario)} palavras)") # Exibe as dimensões da matriz

# vamos visualizar como o dataframe para ficr legivel
df_bow = pd.DataFrame(x_bow.toarray(), columns=vocabulario, index=labels)
print("\nMatriz BoW (cada linha é uma frase, cada coluna é uma palavra):")
print(df_bow) # visualisamos o datafram (df)

Tamanho do vacbulario Bow: 35 palavras únicas
Shape da matriz Bow: (7, 35) (7 frases x 35 palavras)

Matriz BoW (cada linha é uma frase, cada coluna é uma palavra):
            aprendem  artificial  bolo  brincou  cachorro  com  correndo  \
cachorro-0         0           0     0        0         1    0         0   
cachorro-1         0           0     0        0         0    0         1   
cachorro-2         0           0     0        1         0    0         0   
tech-3             0           1     0        0         0    0         0   
tech-4             0           0     0        0         0    0         0   
tech-5             1           0     0        0         0    1         0   
comida-6           0           0     1        0         0    0         0   

            correu  cão  da  ...  ovos  parque  pelo  quintal  receita  redes  \
cachorro-0       1    0   0  ...     0       1     1        0        0      0   
cachorro-1       0    1   0  ...     0       0     1        0   

In [7]:
# Similaridade cossenoidal entre as frases com BoW
"""
Cosine similarity mede o ângulo entre dois vetores.
Vetores apontando na mesma direção = similares (score ~1.0)
Vetores perpendiculares = sem relação (score ~0.0)
"""
sim_bow = cosine_similarity(x_bow)
df_sim_bow = pd.DataFrame(sim_bow, index = labels, columns=labels).round(3)

print("Matriz de similaridade cossenoidal com Bow:")
print(df_sim_bow)

Matriz de similaridade cossenoidal com Bow:
            cachorro-0  cachorro-1  cachorro-2  tech-3  tech-4  tech-5  \
cachorro-0       1.000       0.204       0.000     0.0     0.0     0.0   
cachorro-1       0.204       1.000       0.183     0.0     0.0     0.0   
cachorro-2       0.000       0.183       1.000     0.0     0.0     0.0   
tech-3           0.000       0.000       0.000     1.0     0.0     0.0   
tech-4           0.000       0.000       0.000     0.0     1.0     0.0   
tech-5           0.000       0.000       0.000     0.0     0.0     1.0   
comida-6         0.000       0.000       0.000     0.0     0.0     0.0   

            comida-6  
cachorro-0       0.0  
cachorro-1       0.0  
cachorro-2       0.0  
tech-3           0.0  
tech-4           0.0  
tech-5           0.0  
comida-6         1.0  


In [11]:
# TF e IDF melhoria de bow
"""
Penaliza palavras muito frequentes (como "o", "a", "de")
e valoriza palavras raras e específicas de cada documento.
Mas ainda tem o mesmo problema fundamental: sem sinônimos.
"""

tfidf_vectorizer = TfidfVectorizer()
x_tfidf = tfidf_vectorizer.fit_transform(frases)

sim_tfidf = cosine_similarity(x_tfidf)
df_sim_tfidf = pd.DataFrame(sim_tfidf, index = labels, columns = labels).round(3)

print("Matriz de similaridade cossenoidal com TF-IDF:")
print(df_sim_tfidf)


Matriz de similaridade cossenoidal com TF-IDF:
            cachorro-0  cachorro-1  cachorro-2  tech-3  tech-4  tech-5  \
cachorro-0       1.000       0.155       0.000     0.0     0.0     0.0   
cachorro-1       0.155       1.000       0.137     0.0     0.0     0.0   
cachorro-2       0.000       0.137       1.000     0.0     0.0     0.0   
tech-3           0.000       0.000       0.000     1.0     0.0     0.0   
tech-4           0.000       0.000       0.000     0.0     1.0     0.0   
tech-5           0.000       0.000       0.000     0.0     0.0     1.0   
comida-6         0.000       0.000       0.000     0.0     0.0     0.0   

            comida-6  
cachorro-0       0.0  
cachorro-1       0.0  
cachorro-2       0.0  
tech-3           0.0  
tech-4           0.0  
tech-5           0.0  
comida-6         1.0  


In [16]:
# Visualizando o problema central (análise manual)

# vamos isolar as comparacoes mais reveladoras
pares_interesse = [ # Cada tupla dentro da lista tem 3 elementos
    ("cachorro-0", "cachorro-1", "Mesmo tema, palavras diferentes"), # a: label da primeira frase (referencia a linha no DataFrame)
    ("cachorro-0", "cachorro-2", "Mesmo tema, 1 palavra em comum"), # b: label da segunda frase (referencia a coluna no DataFrame)
    ("tech-3", "tech-4", "Mesmo tema, algumas palavras em comum"),
    ("cachorro-0", "comida-6", "Temas completamente diferentes"),
    ("tech-3", "comida-6", "Temas completamente diferentes"),
]

print("\n=== ANÁLISE DOS PARES CRÍTICOS ===\n")
# :<45 = alinha o texto à esquerda ocupando 45 caracteres (padding)
# :>6  = alinha à direita em 6 caracteres
# :>8  = alinha à direita em 8 caracteres
print(f"{'Par':<45} {'BoW':>6} {'TD-IDF':>8}")
print("-" * 65) # Linha separadora com 65 traços, estética para separar cabeçalho dos dados.

# Itera sobre cada tupla da lista Python desempacota os 3 valores nas variáveis a, b e d
for a, b, descricao in pares_interesse:
    score_bow = df_sim_bow.loc[a,b] # .loc[linha, coluna] acessa uma célula específica do DataFrame pelo nome do índice
    score_tfidf = df_sim_tfidf.loc[a,b]
    print(f"{descricao:<45} {score_bow:>6.3f} {score_tfidf:>8.3f}")

print("\n⚠️  CONCLUSÃO:")
print("Frases com o MESMO SIGNIFICADO mas palavras DIFERENTES")
print("ficam com similaridade BAIXA ou até ZERO.")
print("Isso é o problema fundamental que embeddings semânticos resolvem.")


=== ANÁLISE DOS PARES CRÍTICOS ===

Par                                              BoW   TD-IDF
-----------------------------------------------------------------
Mesmo tema, palavras diferentes                0.204    0.155
Mesmo tema, 1 palavra em comum                 0.000    0.000
Mesmo tema, algumas palavras em comum          0.000    0.000
Temas completamente diferentes                 0.000    0.000
Temas completamente diferentes                 0.000    0.000

⚠️  CONCLUSÃO:
Frases com o MESMO SIGNIFICADO mas palavras DIFERENTES
ficam com similaridade BAIXA ou até ZERO.
Isso é o problema fundamental que embeddings semânticos resolvem.
